# Семинар 03. Строки и простые шифры


## Цели

После семинара вы сможете:

- обрабатывать строки посимвольно;
- реализовывать шифрование и расшифровку по заданному алгоритму;
- проверять обратимость преобразования на граничных случаях.

## Перед началом

Понадобятся строки, циклы, функции, индексация и операция взятия остатка.


## Строки в Python

`str` — неизменяемая последовательность кодовых точек Unicode. Индексация возвращает строку длины один, срез создаёт новую строку, а заменить символ на месте нельзя. Строка в этом смысле похожа на напечатанную этикетку: чтобы исправить одну букву, приходится изготовить новую этикетку, а не менять старую.

```python
text = "Python"
print(text[0])       # P
print(text[1:4])     # yth
print(text[::-1])    # nohtyP
# text[0] = "J"  # TypeError: строки неизменяемы
```

Операция `result += fragment` каждый раз может создавать новую строку и копировать уже накопленный текст. Если добавлять фрагменты по одному, программа снова и снова переписывает весь собранный результат. В цикле это может привести к квадратичному времени. Для большого числа фрагментов их сначала складывают в список, а затем один раз вызывают `"".join(parts)`.

## Unicode и байты

Текст и его байтовое представление — разные вещи. В `str` программа работает с символами, а файл или сеть в конечном счёте принимают байты:

```python
text = "Привет"
payload = text.encode("utf-8")
restored = payload.decode("utf-8")
assert restored == text
```

Кодировка задаёт правила упаковки текста в байты и обратного восстановления. Если записать данные по правилам UTF-8, а прочитать по другим правилам, программа получит ошибку или восстановит не тот текст. Сам набор байтов не сообщает надёжно, какую кодировку выбрал отправитель, поэтому о ней договариваются заранее.

Одна кодовая точка не всегда равна одному видимому символу. Например, `é` можно представить одной кодовой точкой или последовательностью из `e` и отдельного знака ударения. На экране результат выглядит одинаково, но последовательности внутри строки различаются:

```python
import unicodedata

composed = "é"
decomposed = "e\u0301"
assert composed != decomposed
assert unicodedata.normalize("NFC", composed) == unicodedata.normalize("NFC", decomposed)
```

В заданиях преобразуется только фиксированный латинский алфавит. Так границы алгоритма заданы явно, и особенности Unicode не смешиваются с логикой шифра.

## Что здесь называется шифрованием

Симметричное шифрование использует один общий секретный ключ для шифрования и расшифрования. В асимметричной криптографии работает связанная пара ключей: открытый можно распространять, закрытый должен оставаться у владельца.

Открытый ключ получателя позволяет зашифровать данные, которые расшифрует закрытый ключ. В схеме цифровой подписи владелец закрытого ключа создаёт подпись, а открытый ключ позволяет её проверить. Подпись подтверждает автора и целостность, но сама по себе не скрывает содержание сообщения.

Цезарь и Виженер в этом семинаре — упражнения на строки, индексы и остаток от деления. Для защиты реальных данных они бесполезны: оба шифра взламываются без знания ключа. Собственные криптографические схемы почти неизбежно содержат ошибки, поэтому в прикладном коде используют проверенные библиотеки и современные протоколы.

Общий контракт учебных функций:

```python
def encrypt(src: str, key) -> str:
    ...


def decrypt(src: str, key) -> str:
    ...


assert decrypt(encrypt(text, key), key) == text
```

Одного примера недостаточно для проверки обратимости. Нужны как минимум пустая строка, символы вне алфавита, отрицательные и большие сдвиги.

Полезные источники: [Python Unicode HOWTO](https://docs.python.org/3/howto/unicode.html), [строковые операции и стоимость конкатенации](https://docs.python.org/3/library/stdtypes.html#common-sequence-operations), [`unicodedata.normalize`](https://docs.python.org/3/library/unicodedata.html#unicodedata.normalize).


## Шифр Цезаря

Пусть задан алфавит `A` длины `M` и целочисленный сдвиг `offset`. Каждый символ из алфавита заменяется символом с циклически сдвинутым индексом. Символы вне алфавита по условию задания остаются как есть.

Удобно представить алфавит нанесённым на круговой диск: после последней буквы снова идёт первая, а перед первой находится последняя. Поэтому отрицательные и большие сдвиги сводятся к эквивалентному перемещению в пределах одного оборота. Например, для латинского алфавита сдвиги `1`, `27` и `-25` эквивалентны.

При сдвиге `1`: `python → qzuipo`.

Взлом тривиален: для алфавита из 26 букв существует всего 26 различных сдвигов, включая нулевой. Все варианты перебираются мгновенно. Частотный анализ здесь даже не требуется, хотя он работает против более общих моноалфавитных подстановок.

Для алфавита переменной длины полезно заранее построить словарь `символ → индекс`. Вызов `ALPHABET.index(char)` внутри цикла каждый раз линейно ищет символ в алфавите. Для 26 букв разница мала, но при алфавите длины `M` такой поиск добавляет множитель `M`; словарь даёт ожидаемый доступ за `O(1)`.


## Шифр Виженера

Пусть задан алфавит `A` длины `M` и непустой ключ `key` длины `K`. Каждая буква ключа задаёт свой сдвиг, а после последней буквы ключ начинается заново. Ключ можно представить как повторяющуюся ленту, которую протягивают вдоль обрабатываемых букв текста.

Каждую очередную букву текста сдвигают в соответствии с очередной буквой ключа. Для расшифрования выполняют обратный сдвиг. Когда ключ заканчивается, его начинают читать сначала.

В задании символы вне `ALPHABET` не изменяются и **не продвигают ключ**. Лента ключа сдвигается только напротив букв, которые действительно преобразуются. Шифрование и расшифрование обязаны считать позиции одинаково: если одно пропускает пробелы, а другое учитывает их, исходный текст восстановить не получится.

Для алфавита `abcdefghijklmnopqrstuvwxyz` ключ `a` ничего не меняет. Любой однобуквенный ключ превращает Виженера в Цезаря. Ключ `ab` оставляет буквы на чётных обрабатываемых позициях без изменения, а буквы на нечётных сдвигает на один: `python → pztioo`.

Виженер устроен сложнее Цезаря, но повторяющийся ключ создаёт периодическую структуру. Длину ключа и сдвиги восстанавливают статистическими методами, а короткий ключ можно перебрать. Для современной защиты этот шифр непригоден.

При заранее построенной таблице индексов оба преобразования проходят текст один раз: `O(N)` по времени. Результат содержит `N` символов, поэтому требует `O(N)` памяти — строку всё равно нужно создать заново.


## Не путайте разные преобразования

| Операция | Обратима | Нужен секрет | Назначение |
|---|:---:|:---:|---|
| кодирование, например UTF-8 или Base64 | да | нет | представить данные в другом формате |
| шифрование | да, при наличии ключа | да | скрыть содержание |
| криптографическое хеширование | нет | нет | получить фиксированный отпечаток данных |
| цифровая подпись | проверяется открытым ключом | закрытый ключ нужен для создания | подтвердить автора и целостность |

Base64 ничего не шифрует: любой получатель может декодировать данные без ключа. Обычный `hash()` Python тоже не является криптографическим хешем и для некоторых типов меняется между запусками процесса. Эти операции решают разные задачи; если назвать кодирование шифрованием, защищённости от этого не появится.


## Самопроверка

1. Почему изменение одного символа строки создаёт новую строку?
2. Чем `str` отличается от `bytes`, и где участвует кодировка?
3. Почему две визуально одинаковые Unicode-строки могут не пройти проверку `==`?
4. Зачем в формуле Цезаря нужен остаток от деления?
5. Почему символы вне алфавита должны одинаково влиять на позицию ключа при шифровании и расшифровании?
6. Чем кодирование, шифрование, хеширование и подпись отличаются по цели и обратимости?


## Итоги

- `str` хранит Unicode-текст, `bytes` — байты; кодировка переводит одно в другое.
- Строки неизменяемы, поэтому результат посимвольного преобразования приходится собирать заново.
- Повторная конкатенация в цикле может дать квадратичное время; список и `join()` дают линейную сборку.
- Остаток от деления реализует циклический сдвиг по алфавиту.
- Обратимость требует единого контракта обработки каждого символа.
- Цезарь и Виженер годятся для тренировки алгоритмов, но не для защиты данных.


## Задание 1. Шифр Цезаря (1 балл)

Реализуйте шифрование и расшифровку строчных латинских букв. Символы вне `ALPHABET` оставляйте без изменений.

```python
ALPHABET = "abcdefghijklmnopqrstuvwxyz"

def caesar_encrypt(src: str, offset: int) -> str:
    ...

def caesar_decrypt(src: str, offset: int) -> str:
    ...
```

**Примеры**

```python
caesar_encrypt("python", 1) == "qzuipo"
caesar_decrypt("qzuipo", 1) == "python"
```

**Критерии проверки:** поддерживаются отрицательные и большие сдвиги; для любой строки выполняется `decrypt(encrypt(text, key), key) == text`.


## Задание 2. Шифр Виженера (2 балла)

Реализуйте шифрование и расшифровку строчных латинских букв по непустому ключу. Символы вне `ALPHABET` оставляйте без изменений и не учитывайте при продвижении по ключу.

```python
def vigenere_encrypt(src: str, key: str) -> str:
    ...

def vigenere_decrypt(src: str, key: str) -> str:
    ...
```

**Примеры**

```python
vigenere_encrypt("python", "ab") == "pztioo"
vigenere_decrypt("pztioo", "ab") == "python"
```

**Критерии проверки:** пустой ключ отклоняется с `ValueError`; преобразование обратимо; исходные аргументы не изменяются.


## Бонусное задание. Сравнение строк с Backspace

Символ `#` означает Backspace. Удаление из пустой строки оставляет её пустой. Определите, совпадут ли две строки после обработки всех Backspace.

```python
def backspace_compare(left: str, right: str) -> bool:
    ...
```

**Примеры**

```python
backspace_compare("ab#c", "ad#c") is True   # обе строки превратятся в "ac"
backspace_compare("ab##", "c#d#") is True  # обе строки станут пустыми
backspace_compare("a#c", "b") is False
```

**Ограничения:** строки содержат строчные латинские буквы и `#`, длина каждой строки — от 1 до 200.

**Оценивание:** 1 балл за корректное решение; 2 балла за `O(N)` по времени и `O(1)` по дополнительной памяти.
